# Rate Limiting and Performance

This notebook shows how to handle rate limits and optimize performance.

## Automatic Rate Limiting

In [ ]:
import asyncio
from knowledge_lookup import CentralKnowledgeLookup

lookup = CentralKnowledgeLookup()

# Library automatically handles rate limits
# Multiple queries are queued and executed with appropriate delays

result = await lookup.search_concepts(
    query="diabetes",
    sources=["BioPortal", "UMLS", "ChEMBL"],
    limit=20
)

print(f"Results: {len(result.concepts)} concepts")

await lookup.close()

## Manual Rate Limiting

In [ ]:
import asyncio
import time
from knowledge_lookup import CentralKnowledgeLookup, LookupConfig

config = LookupConfig(
    timeout_per_source=10.0,
    max_results_per_source=10
)

lookup = CentralKnowledgeLookup(config)

# Query sources sequentially with delays
sources = ["BioPortal", "UMLS", "ChEMBL"]

for source in sources:
    start_time = time.time()
    try:
        result = await lookup.search_concepts(
            query="diabetes",
            sources=[source],
            limit=10
        )
        elapsed = time.time() - start_time
        print(f"{source}: {len(result.concepts)} results in {elapsed:.2f}s")
        
        # Add delay between requests
        await asyncio.sleep(1)
    except Exception as e:
        print(f"{source} error: {e}")

await lookup.close()

## Retry with Backoff

In [ ]:
import asyncio
import random
from knowledge_lookup import CentralKnowledgeLookup

async def search_with_backoff(lookup, query, sources, max_retries=3):
    """Search with exponential backoff."""
    for attempt in range(max_retries):
        try:
            return await lookup.search_concepts(query, sources=sources)
        except Exception as e:
            if 'rate' in str(e).lower() and attempt < max_retries - 1:
                wait_time = 2 ** attempt + random.uniform(0, 1)
                print(f"Rate limited. Waiting {wait_time:.2f}s...")
                await asyncio.sleep(wait_time)
            else:
                raise

async def retry_example():
    lookup = CentralKnowledgeLookup()
    
    result = await search_with_backoff(
        lookup,
        "diabetes",
        sources=["BioPortal"],
        max_retries=5
    )
    
    print(f"Results: {len(result.concepts)}")
    await lookup.close()

await retry_example()

## Caching for Performance

In [ ]:
import asyncio
import time
from knowledge_lookup import CentralKnowledgeLookup, LookupConfig

# With caching
config_with_cache = LookupConfig(
    cache_enabled=True,
    cache_ttl=3600,
    cache_dir="./cache"
)

lookup = CentralKnowledgeLookup(config_with_cache)

# First search - hits API
start = time.time()
result1 = await lookup.search_concepts("diabetes", sources=["BioPortal"])
first_time = time.time() - start

# Second search - uses cache
start = time.time()
result2 = await lookup.search_concepts("diabetes", sources=["BioPortal"])
second_time = time.time() - start

print(f"First search (API): {first_time:.2f}s, {len(result1.concepts)} results")
print(f"Second search (cached): {second_time:.2f}s, {len(result2.concepts)} results")

await lookup.close()

## Best Practices

In [ ]:
# 1. Use API keys for higher rate limits
# 2. Enable caching for repeated queries
# 3. Implement exponential backoff for retries
# 4. Use smaller limits first, then paginate
# 5. Monitor rate limit headers

# Example: Batch queries instead of multiple single queries
# Instead of:
# for term in terms:
#     result = await lookup.search_concepts(term)

# Use:
# result = await lookup.search_concepts(" OR ".join(terms))

# Or use appropriate caching with TTL based on source update frequency

## Next Steps

- See `docs/guides/rate_limiting.md` for detailed rate limiting guide
- See `docs/guides/caching.md` for caching configuration